# user

In [ ]:
VERSION = "2.1.1"
# PSF_NAME = "psf_style-airy_size-41_pixelscale-0.1"
PSF_NAME = "psf_style-airy_size-81_pixelscale-0.06"
FILTER_NAME = "gauss_3.0_7x7.conv"
PIXEL_SCALE = 0.1

# lib

In [ ]:
import numpy
from pandas import DataFrame, read_csv
from tqdm import tqdm

from plots import (
    ldac,
    gauss_conv,
    hdul_object_ldac,
    extra_psf_images,
    extra_psf_images_reshaped,
    image_fitting,
    case_result,
    histos,
    ellips_se,
    ellips_model,
    case_result_param,
    delete_duplicate,
    tfds_info,
    slice_csv,
    DIRS,
    background,
)

DATASET_DIR_ORIGIN = DIRS["pisco_euclid"] / VERSION
DATASET_DIR = DIRS["pisco_euclid_se"] / VERSION
PSF_DIR = DIRS["RESULTS_SEXTRACTOR"] / PSF_NAME
RESULTS_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION
CATALOG_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION / "CATALOG"
IMAGES_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION / "IMAGES"
PLOTS_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# info databases

In [ ]:
info = tfds_info(style="pisco_euclid", version=VERSION, data_dir=DIRS["DATASETS"])
for key, value in info.release_notes.items():
    print(f"{key}: {value}")

# filter

In [ ]:
gauss_conv(DIRS["psf_se"] / FILTER_NAME, PLOTS_DIR)

# ldac

In [ ]:
objects_data = ldac(PSF_DIR, PSF_NAME)

In [ ]:
shape_vignet = hdul_object_ldac(objects_data, PLOTS_DIR)

# extra images psf

In [ ]:
extra_psf_images(DIRS["psf_se"], PSF_NAME, PSF_DIR, PLOTS_DIR, loggy=False)
extra_psf_images(DIRS["psf_se"], PSF_NAME, PSF_DIR, PLOTS_DIR, loggy=True)

In [ ]:
extra_psf_images_reshaped(PSF_DIR, PSF_NAME, shape_vignet, PLOTS_DIR, style="samples")
extra_psf_images_reshaped(PSF_DIR, PSF_NAME, shape_vignet, PLOTS_DIR, style="residuals")

# extra

In [ ]:
CHOICE = "000003_simu"

In [ ]:
image_fitting(DATASET_DIR, IMAGES_DIR, CHOICE, PLOTS_DIR)
image_fitting(DATASET_DIR, IMAGES_DIR, CHOICE, PLOTS_DIR, transformed=True)

In [ ]:
background(IMAGES_DIR / f"back_{CHOICE}")


# inference

In [ ]:
PRED_FILES = sorted(list((CATALOG_DIR).glob("*_simu.csv")))
TRUE_FILES = sorted(list((DATASET_DIR).glob("*_true_galaxies_params.csv")))

all_obs_e1_model_err, all_obs_e2_model_err, all_obs_e1_model, all_obs_e2_model = [], [], [], []
all_obs_ell_se, all_obs_theta_se = [], []
avg_obs_e1_model, avg_obs_e2_model, avg_obs_e1_se, avg_obs_e2_se = [], [], [], []

mag, mag_err, sersic, sersic_err, hlr, true_mag, true_hlr, true_sersic = [[] for _ in range(8)]
slice_sersic, slice_sersic_err, slice_mag, slice_mag_err, slice_hlr = [[] for _ in range(5)]
slice_true_hlr, slice_true_mag, slice_true_sersic = [[] for _ in range(3)]
detections = []

TOTAL = len(PRED_FILES)
for result_file, true_file in tqdm(zip(PRED_FILES, TRUE_FILES), total=TOTAL):
    df_pred = read_csv(result_file).sort_values("X_IMAGE")
    df_true = read_csv(true_file).sort_values("x")

    obs_e1_se, obs_e2_se = ellips_se(df_pred["ELLIPTICITY"], df_pred["THETA_IMAGE"])
    avg_obs_e1_se.append(obs_e1_se)
    avg_obs_e2_se.append(obs_e2_se)

    all_obs_ell_se.append(df_pred["ELLIPTICITY"].to_numpy())
    all_obs_theta_se.append(df_pred["THETA_IMAGE"].to_numpy())

    obs_e1_model, obs_e2_model = ellips_model(
        df_pred["ELLIP1ERRMODEL_IMAGE"],
        df_pred["ELLIP2ERRMODEL_IMAGE"],
        df_pred["ELLIP1MODEL_IMAGE"],
        df_pred["ELLIP2MODEL_IMAGE"],
    )
    avg_obs_e1_model.append(obs_e1_model)
    avg_obs_e2_model.append(obs_e2_model)

    all_obs_e1_model.append(df_pred["ELLIP1MODEL_IMAGE"].to_numpy())
    all_obs_e2_model.append(df_pred["ELLIP2MODEL_IMAGE"].to_numpy())
    all_obs_e1_model_err.append(df_pred["ELLIP1ERRMODEL_IMAGE"].to_numpy())
    all_obs_e2_model_err.append(df_pred["ELLIP2ERRMODEL_IMAGE"].to_numpy())

    mag.append(df_pred["MAG_AUTO"].to_numpy())
    mag_err.append(df_pred["MAGERR_AUTO"].to_numpy())

    sersic.append(df_pred["SPHEROID_SERSICN"].to_numpy())
    sersic_err.append(df_pred["SPHEROID_SERSICNERR"].to_numpy())

    hlr.append(df_pred["FLUX_RADIUS"].to_numpy())

    true_mag.append(df_true["mag"].to_numpy())
    true_sersic.append(df_true["sersic"].to_numpy())
    true_hlr.append(df_true["hlr"].to_numpy())

    NUMBER_DETECTED = df_pred.shape[0]
    detections.append(NUMBER_DETECTED)

    if NUMBER_DETECTED != 10:
        df_pred = delete_duplicate(df_pred)
        # df_true = df_true.filter(df_true["x"].is_in(df_pred["assoc1"]))
        df_true = df_true[df_true["x"].isin(df_pred["assoc1"])]
        assert df_pred.shape[0] == df_true.shape[0]

    slice_mag.append(df_pred["MAG_AUTO"].to_numpy())
    slice_mag_err.append(df_pred["MAGERR_AUTO"].to_numpy())

    slice_sersic.append(df_pred["SPHEROID_SERSICN"].to_numpy())
    slice_sersic_err.append(df_pred["SPHEROID_SERSICNERR"].to_numpy())

    slice_hlr.append(df_pred["FLUX_RADIUS"].to_numpy())

    slice_true_mag.append(df_true["mag"].to_numpy())
    slice_true_sersic.append(df_true["sersic"].to_numpy())
    slice_true_hlr.append(df_true["hlr"].to_numpy())

detections = numpy.array(detections)

all_obs_ell_se = numpy.concatenate(all_obs_ell_se)
all_obs_theta_se = numpy.concatenate(all_obs_theta_se)
all_obs_e1_model = numpy.concatenate(all_obs_e1_model)
all_obs_e2_model = numpy.concatenate(all_obs_e2_model)
all_obs_e1_model_err = numpy.concatenate(all_obs_e1_model_err)
all_obs_e2_model_err = numpy.concatenate(all_obs_e2_model_err)

sersic = numpy.concatenate(sersic)
sersic_err = numpy.concatenate(sersic_err)
true_sersic = numpy.concatenate(true_sersic)
slice_sersic = numpy.concatenate(slice_sersic)
slice_sersic_err = numpy.concatenate(slice_sersic_err)
slice_true_sersic = numpy.concatenate(slice_true_sersic)

mag = numpy.concatenate(mag)
mag_err = numpy.concatenate(mag_err)
true_mag = numpy.concatenate(true_mag)
slice_mag = numpy.concatenate(slice_mag)
slice_mag_err = numpy.concatenate(slice_mag_err)
slice_true_mag = numpy.concatenate(slice_true_mag)

hlr = numpy.concatenate(hlr)
true_hlr = numpy.concatenate(true_hlr)
slice_hlr = numpy.concatenate(slice_hlr)
slice_true_hlr = numpy.concatenate(slice_true_hlr)

loss_comp = round(((sersic.shape[0] - slice_sersic.shape[0]) / sersic.shape[0]) * 100, 2)
loss_gal = round(((sersic.shape[0]) / true_sersic.shape[0]) * 100, 2)

DataFrame(
    data={
        "avg_obs_e1_se": avg_obs_e1_se,
        "avg_obs_e2_se": avg_obs_e2_se,
        "avg_obs_e1_model": avg_obs_e1_model,
        "avg_obs_e2_model": avg_obs_e2_model,
    },
).to_csv(RESULTS_DIR / "pred_avg_obs_e.csv", index=False)

print("normal", sersic.shape[0], sersic_err.shape[0], true_sersic.shape[0])
print("slice", slice_sersic.shape[0], slice_sersic_err.shape[0], slice_true_sersic.shape[0])

del (
    df_pred,
    df_true,
    NUMBER_DETECTED,
    avg_obs_e1_model,
    avg_obs_e2_model,
    avg_obs_e1_se,
    avg_obs_e2_se,
    obs_e1_model,
    obs_e1_se,
    obs_e2_model,
    obs_e2_se,
    PRED_FILES,
    TRUE_FILES,
)

# loading params

In [ ]:
true_images_params = slice_csv(DATASET_DIR_ORIGIN / "images_parameters_true.csv", TOTAL)
true_avg_obs_e = true_images_params[["avg_obs_e1", "avg_obs_e2"]].to_numpy()
true_avg_int_e = true_images_params[["avg_int_e1", "avg_int_e2"]].to_numpy()
true_avg_g = true_images_params[["avg_g1", "avg_g2"]].to_numpy()

pred_avg_obs_e = slice_csv(RESULTS_DIR / "pred_avg_obs_e.csv", TOTAL)
sex_pred_obs_e = pred_avg_obs_e[["avg_obs_e1_se", "avg_obs_e2_se"]].to_numpy()
model_pred_obs_e = pred_avg_obs_e[["avg_obs_e1_model", "avg_obs_e2_model"]].to_numpy()

print(true_avg_obs_e.shape, true_avg_int_e.shape, true_avg_g.shape, pred_avg_obs_e.shape)
del pred_avg_obs_e, true_images_params

# histo

In [ ]:
bins = 50
loggy = False
histos(
    detections,
    sersic,
    sersic_err,
    true_sersic,
    mag,
    mag_err,
    true_mag,
    hlr,
    true_hlr,
    sex_pred_obs_e,
    model_pred_obs_e,
    true_avg_obs_e,
    PLOTS_DIR,
    PIXEL_SCALE,
    bins=bins,
    loggy=loggy,
)

# results

In [ ]:
case_result(true_avg_obs_e, true_avg_g, PLOTS_DIR / "results_ell_best.pdf")
case_result(sex_pred_obs_e, true_avg_g, PLOTS_DIR / "results_ell_internal.pdf")
case_result(model_pred_obs_e, true_avg_g, PLOTS_DIR / "results_ell_model.pdf")

# results by params

In [ ]:
case_result_param(
    slice_mag, slice_true_mag, loss_comp, loss_gal, PLOTS_DIR, symbol="mag", err=slice_mag_err
)
case_result_param(
    slice_hlr, slice_true_hlr, loss_comp, loss_gal, PLOTS_DIR, symbol="hlr", pixel_scale=PIXEL_SCALE
)
case_result_param(
    slice_sersic,
    slice_true_sersic,
    loss_comp,
    loss_gal,
    PLOTS_DIR,
    symbol="sersic",
    err=slice_sersic_err,
)